# Retos del workshop RAG · notebook de arranque

Este notebook trae el punto de partida de algunos retos. Los bloques marcados con `TODO` son tuyos. Copia este archivo a `notebooks/04_challenges.ipynb` en tu repo.

## C1 · HNSW contra fuerza bruta con FAISS

Instala FAISS y ejecuta la línea base. Después completa los TODO.

In [ ]:
%pip install -q faiss-cpu matplotlib

In [ ]:
import time, faiss, numpy as np
import matplotlib.pyplot as plt
faiss.omp_set_num_threads(1)
d, nq, k = 64, 200, 10

def recall_at_k(I, T): return float(np.mean([len(set(a) & set(b)) / len(b) for a, b in zip(I, T)]))

def run(n, Ms=(8, 16, 32), efs=(10, 16, 32, 64, 128, 256)):
    rng = np.random.default_rng(42)
    xb = rng.random((n, d), dtype="float32"); xq = rng.random((nq, d), dtype="float32")
    flat = faiss.IndexFlatL2(d); flat.add(xb)
    t = time.time(); _, T = flat.search(xq, k); t_flat = (time.time() - t) / nq * 1000
    rows = []
    for M in Ms:
        ix = faiss.IndexHNSWFlat(d, M); ix.hnsw.efConstruction = 100
        t = time.time(); ix.add(xb); build = time.time() - t
        for ef in efs:
            ix.hnsw.efSearch = ef
            t = time.time(); _, I = ix.search(xq, k); ms = (time.time() - t) / nq * 1000
            rows.append(dict(n=n, M=M, ef=ef, recall=recall_at_k(I, T), ms=ms, build=build, flat_ms=t_flat))
    return rows

res = run(20_000)
for r in res: print(f"M={r['M']:>2} ef={r['ef']:>3} recall@{k}={r['recall']:.3f} {r['ms']:.3f} ms/q (build {r['build']:.1f}s)  exacto={r['flat_ms']:.3f} ms/q")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for M in (8, 16, 32):
    s = [x for x in res if x["M"] == M]
    ax[0].plot([x["ef"] for x in s], [x["recall"] for x in s], "o-", label=f"M={M}")
    ax[1].plot([x["ms"] for x in s], [x["recall"] for x in s], "o-", label=f"M={M}")
ax[0].set_xscale("log", base=2); ax[0].set_xlabel("efSearch"); ax[0].set_ylabel("recall@10")
ax[0].axhline(0.95, ls="--", c="gray"); ax[0].legend()
ax[1].set_xlabel("ms/consulta"); ax[1].set_ylabel("recall@10")
ax[1].axvline(res[0]["flat_ms"], ls="--", c="r", label="exacto"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Barrido de n (M=16). Tarda unos minutos con n=500_000.
for n in (20_000, 100_000, 500_000):
    for ef in (64, 256):
        r = run(n, Ms=(16,), efs=(ef,))[0]
        print(f"n={n:>7,} ef={ef:>3}: recall={r['recall']:.3f}  HNSW {r['ms']:.3f} ms/q vs exacto {r['flat_ms']:.3f} ms/q  (build {r['build']:.1f}s)")

In [ ]:
# TODO 1: grafica recall vs efSearch (una curva por M) con matplotlib.
# TODO 2: repite el experimento con n = 100_000 y n = 500_000. ¿Desde qué n HNSW supera al exacto?
# TODO 3: escribe tus conclusiones en RESPUESTAS.md

## C4 · Evaluación automática de la recuperación

Necesitas las variables del notebook 03 (`retriever`, `all_docs`, …) o reconstruirlas aquí.

In [ ]:
import csv
rows = list(csv.DictReader(open("eval/preguntas.csv", encoding="utf-8")))

def hit_at_k(retriever, rows):
    hits = 0
    for r in rows:
        docs = retriever.invoke(r["pregunta"])
        hits += any(r["palabra_clave"].lower() in d.page_content.lower()
                    and d.metadata["source"] == r["fuente_esperada"] for d in docs)
    return hits / len(rows)

for cs, ov in [(400, 52), (900, 120), (1500, 195)]:
    store, n = build_store(cs, ov)
    out = {}
    for k in (3, 5):
        _, retr = build_chain(store, k=k, threshold=0.0)   # umbral 0 para medir solo ranking
        out[k] = hit_at_k(retr, rows)
    print(f"chunk_size={cs} ({n} chunks): hit@3={out[3]:.0%}  hit@5={out[5]:.0%}")


## C6 · Persistencia del índice en disco

Punto de partida: cambia el cliente en memoria por uno con ruta.

In [ ]:
import time
from pathlib import Path
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(path="./qdrant_data")          # (en Colab: monta Drive y usa esa ruta)
t0 = time.time()
if not client.collection_exists(COLLECTION):
    client.create_collection(COLLECTION, vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE))
    vectorstore = QdrantVectorStore(client=client, collection_name=COLLECTION, embedding=embeddings)
    vectorstore.add_documents(docs)
    print(f"indexado en {time.time()-t0:.1f}s")
else:
    vectorstore = QdrantVectorStore(client=client, collection_name=COLLECTION, embedding=embeddings)
    print(f"reabierto en {time.time()-t0:.2f}s")
